# WINDOW FUNCTIONS

In [0]:
# ============================================================
# 08_WINDOW_FUNCTIONS
#
# Purpose:
# Learn and implement Spark SQL Window Functions
#
# Input:
# ecommerce.silver
#
# Output:
# ecommerce.gold window analytics tables
#
# Functions:
# - ROW_NUMBER
# - RANK
# - DENSE_RANK
# - LAG
# - LEAD
# - Running totals
#
# ============================================================


from pyspark.sql.functions import *
from pyspark.sql.window import Window



# ============================================================
# 1. Configuration
# ============================================================


catalog = "ecommerce"


silver_schema = "silver"

gold_schema = "gold"



# ============================================================
# 2. Read Silver Tables
# ============================================================


customers = spark.table(
    "ecommerce.silver.customers"
)


orders = spark.table(
    "ecommerce.silver.orders"
)


products = spark.table(
    "ecommerce.silver.products"
)





# ============================================================
# 3. ROW_NUMBER()
#
# Use Case:
# Find latest order of every customer
#
# Example:
#
# Customer 1:
# order 101  Jan
# order 102  Feb
# order 103  March
#
# Keep order 103
#
# ============================================================


customer_order_window = (

    Window

    .partitionBy(
        "customer_id"
    )

    .orderBy(
        col("order_date").desc()
    )

)



latest_customer_orders = (

    orders

    .withColumn(

        "row_number",

        row_number()
        .over(customer_order_window)

    )


    .filter(

        col("row_number")==1

    )


    .drop(
        "row_number"
    )

)



latest_customer_orders.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.latest_customer_orders"
)



print("ROW_NUMBER completed")





# ============================================================
# 4. RANK()
#
# Rank customers by total spending
#
# Example:
#
# Ali       50000 Rank 1
# Ahmed     40000 Rank 2
# Sara      40000 Rank 2
#
# ============================================================



customer_spending = (

    orders

    .groupBy(
        "customer_id"
    )

    .agg(

        sum("total_amount")
        .alias("total_spent")

    )

)



rank_window = (

    Window

    .orderBy(
        col("total_spent").desc()
    )

)



customer_rank = (

    customer_spending

    .withColumn(

        "customer_rank",

        rank()
        .over(rank_window)

    )



    .join(

        customers,

        "customer_id",

        "left"

    )

)



customer_rank.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.customer_rank"
)



print("RANK completed")





# ============================================================
# 5. DENSE_RANK()
#
# Rank products by revenue
#
# Difference:
#
# RANK:
# 1
# 2
# 2
# 4
#
# DENSE_RANK:
# 1
# 2
# 2
# 3
#
# ============================================================



product_sales = (

    orders


    .groupBy(
        "product_id"
    )


    .agg(

        sum("total_amount")
        .alias("revenue")

    )

)



product_rank_window = Window.orderBy(
    col("revenue").desc()
)



product_revenue_rank = (

    product_sales


    .withColumn(

        "product_rank",

        dense_rank()
        .over(product_rank_window)

    )


    .join(

        products,

        "product_id",

        "left"

    )

)



product_revenue_rank.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.product_rank"
)



print("DENSE_RANK completed")





# ============================================================
# 6. LAG()
#
# Compare current month revenue
# with previous month revenue
#
# Example:
#
# Jan 1000
# Feb 1500
#
# Difference:
# +500
#
# ============================================================



monthly_sales = (

    orders


    .withColumn(

        "month",

        date_format(
            col("order_date"),
            "yyyy-MM"
        )

    )


    .groupBy(
        "month"
    )


    .agg(

        sum("total_amount")
        .alias("monthly_revenue")

    )

)



lag_window = Window.orderBy(
    "month"
)



monthly_comparison = (

    monthly_sales


    .withColumn(

        "previous_month_revenue",

        lag(
            "monthly_revenue"
        )
        .over(lag_window)

    )


    .withColumn(

        "growth",

        col("monthly_revenue")
        -
        col("previous_month_revenue")

    )

)



monthly_comparison.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.monthly_growth"
)



print("LAG completed")





# ============================================================
# 7. LEAD()
#
# Find customer's next order date
#
# Example:
#
# Order 1
# Next order = Order 2
#
# ============================================================



customer_order_window = (

    Window

    .partitionBy(
        "customer_id"
    )

    .orderBy(
        "order_date"
    )

)



next_purchase = (

    orders


    .withColumn(

        "next_order_date",

        lead(
            "order_date"
        )
        .over(customer_order_window)

    )

)



next_purchase.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.customer_next_purchase"
)



print("LEAD completed")





# ============================================================
# 8. Running Total
#
# Cumulative revenue over time
#
# Example:
#
# Jan 1000
# Feb 2000
#
# Running:
#
# Jan 1000
# Feb 3000
#
# ============================================================



running_window = (

    Window

    .orderBy(
        "order_date"
    )

    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )

)



running_sales = (

    orders


    .select(

        "order_date",

        "total_amount"

    )


    .withColumn(

        "running_revenue",

        sum("total_amount")
        .over(running_window)

    )

)



running_sales.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable(
    "ecommerce.gold.running_sales"
)



print("Running Total completed")





# ============================================================
# 9. Validation
# ============================================================


spark.sql("""
SHOW TABLES IN ecommerce.gold
""").show()

ROW_NUMBER completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


RANK completed
DENSE_RANK completed
LAG completed
LEAD completed
Running Total completed
+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|    gold|  analytics_category|      false|
|    gold|analytics_custome...|      false|
|    gold|analytics_executi...|      false|
|    gold|   analytics_payment|      false|
|    gold|   analytics_returns|      false|
|    gold|analytics_revenue...|      false|
|    gold|analytics_top_cus...|      false|
|    gold|analytics_top_pro...|      false|
|    gold|customer_next_pur...|      false|
|    gold|       customer_rank|      false|
|    gold|    customer_summary|      false|
|    gold|latest_customer_o...|      false|
|    gold|      monthly_growth|      false|
|    gold|       monthly_sales|      false|
|    gold|    payment_analysis|      false|
|    gold| product_performance|      false|
|    gold|        product_rank|      false|
|    gold|     return_analysis|